# 02 · Interferential Current Stimulation

**Physics of Electrical Neurostimulation** ·  1st FALAN Latin American Training Program in Neuroscience, Santiago, August 2026

Leonel Medina · Rodrigo Osorio · Cristian Morales — [NeuroEng@USACH](https://www.neuroeng-usach.cl), Universidad de Santiago de Chile

| | |
|---|---|
| Notebook 01 | **Transcutaneous stimulation** — the field, the axon, the strength-duration curve |
| Notebook 02 | **Interferential current** — two carriers, one beat, deep activation |
| Lab | TENS/EMS unit on your own forearm; handouts in `handouts/` |
| No install | `explorer/index.html` runs the two core figures in any browser |

### What you will be able to do by the end
1. Explain how two medium-frequency currents interfere to produce a low-frequency amplitude-modulated beat *inside* tissue.
2. Locate where the beat is deepest for a given montage, and set up a crossed (quadripolar) montage on a forearm.
3. Elicit pulsatile finger-muscle contractions with a carrier that is itself far too fast for an axon to follow cycle by cycle.
4. Demonstrate block of cutaneous afferents under a kHz field, and say why the same field can block small superficial fibres while driving deeper motor axons.
5. Contrast all of this with the single-pulse stimulation of Notebook 01.

### How the 70 minutes are spent
| | |
|---|---|
| Part 1-2 — what interference is, and the beat at one point | ~15 min |
| Part 3 — the field of four electrodes, and where the beat is deepest | ~20 min |
| Part 4 — does a fibre actually fire? | ~10 min |
| Lab — crossed montage, motor threshold, afferent block | ~25 min |

> **Prerequisite:** Notebook 01, Module 0. The volume conductor here is literally the same object.

## 0 · Setup

The same bootstrap cell as Notebook 01 — if you already ran that notebook in this session, this is instant.

In [ ]:
# One-time setup: fetches the setup script if needed and runs it. On Colab it also
# installs NEURON, which requires the kernel to restart once -- if that happens,
# just run this cell again. Curious what it does? Open notebooks/setup_workshop.py
import pathlib, urllib.request
URL = ("https://raw.githubusercontent.com/neuroeng-usach/"
       "falan-neurostim-workshop/main/notebooks/setup_workshop.py")
if not pathlib.Path("setup_workshop.py").exists():
    urllib.request.urlretrieve(URL, "setup_workshop.py")
%run -i setup_workshop.py

## What is actually being computed here

This notebook adds **no new physics**. Every potential value is still a sum of
`layered_field._point_potential` calls — the same validated two-layer volume conductor you
explored in Notebook 01's Module 0. `src/interferential_field.py` only adds two things:

1. **3-D electrode placement** on the skin surface, so four pads can sit at arbitrary
   `(x, y)` rather than along a single line — that is what makes the montage "crossed".
2. **The interference arithmetic** that turns two static current-density fields into the
   low-frequency beat a nerve can actually respond to.

Parts 1-3 are pure NumPy and run anywhere. Part 4 drives a real MRG axon and needs
NEURON/PyFibers.

## Part 1 — What interferential stimulation is

**Interferential current (IFC)** therapy delivers a *medium-frequency* current
(typically **1–10 kHz**, most often ~4 kHz) rather than a low-frequency pulse
train. Two independent channels are used, at slightly different frequencies:

$$ f_1 = f_c - \tfrac{\Delta f}{2}, \qquad f_2 = f_c + \tfrac{\Delta f}{2}, \qquad \Delta f = |f_1 - f_2| $$

Where the two channels' currents overlap *inside the tissue*, they superpose
into a carrier at ~$f_c$ whose **amplitude is modulated at the beat frequency**
$\Delta f$ (the *amplitude-modulated frequency*, AMF). That slow beat —
typically **1–250 Hz**, chosen for the therapeutic goal — is the effective
low-frequency stimulus, generated in the tissue instead of applied at the skin.

**Why bother? Three reasons:**

1. **Skin comfort.** The skin behaves largely capacitively, so its impedance
   falls with frequency, roughly $Z_{skin} \sim \dfrac{1}{2\pi f\,C}$. A 4 kHz
   carrier meets far lower impedance than a 100 Hz pulse train would, so more
   current reaches deep tissue for the same (comfortable) surface voltage — less
   of the sharp, "biting" sensation low-frequency surface stimulation causes.
2. **The nerve feels the beat, not the carrier.** A nerve membrane is a
   low-pass filter (its time constant is on the order of the chronaxie, ~0.1–0.5
   ms). It cannot follow a 4 kHz carrier cycle-by-cycle, but it *does* respond to
   the ~100 Hz swelling and fading of that carrier's amplitude — the beat.
3. **Depth / targeting.** With four electrodes arranged so the two channels
   *cross*, the region of strongest amplitude modulation sits deep between the
   pads, and can be "steered" by changing the balance of the two channel
   currents — the basis of the quadripolar (true interferential) technique
   modelled in Part 3.

*Premodulated (bipolar) vs. true interferential (quadripolar):* some devices
pre-mix the two frequencies and deliver the already-modulated signal through a
single pad pair. That is convenient but the modulation is fixed in shape. The
**true interferential** setup — two separate channels, four electrodes, mixing
*in the tissue* — is what produces the spatial modulation pattern we visualise
here.

### Mathematical background: the beat

At a point where channel A contributes an oscillating current of amplitude $A$
(at $f_1$) and channel B contributes amplitude $B$ (at $f_2$), the total is a
sum of two cosines:

$$ I(t) = A\cos(2\pi f_1 t) + B\cos(2\pi f_2 t). $$

This is a fast carrier at the mean frequency $f_c$ multiplied by a slowly
varying envelope at the beat frequency $\Delta f$. Its upper envelope is

$$ E(t) = \sqrt{A^2 + B^2 + 2AB\cos(2\pi\,\Delta f\, t)}, $$

which swings between a **maximum $A+B$** (the two carriers momentarily in phase)
and a **minimum $|A-B|$** (in antiphase). The fraction of the carrier that is
modulated — the **modulation depth** — is therefore

$$ m = \frac{(A+B) - |A-B|}{(A+B) + |A-B|} = \frac{\min(A,B)}{\max(A,B)}\cdot\!\Big|_{\text{(equal signs)}}, \qquad 0 \le m \le 1. $$

The key point: $m = 100\%$ **only where the two channels contribute equally**
($A=B$), and falls off as one channel dominates. Part 2 lets you feel this with
sliders; Part 3 shows what it means once $A$ and $B$ become *vectors* that vary
from point to point in the tissue.

## Part 2 — The beat at a single point

**Before you run it:** the two carriers are only ~2.5% apart in frequency
(4000 vs 4100 Hz). Do you expect their sum to look like a plain 4 kHz tone, or
something slower? Run and see.

**Then try this:**
- Set the **channel amplitudes equal** ($A=B$) and note the modulation depth.
  Now make them unequal. What happens to the depth of the beat (the difference
  between the envelope's peaks and troughs)? When does the beat disappear?
- Change the **beat frequency** from 100 Hz to, say, 5 Hz and 250 Hz — this is
  the therapeutic AMF you would dial for different goals. The carrier barely
  changes; the *envelope* is what slows down or speeds up.

In [ ]:
panel_beat = ws.Panel(
    ip.draw_beat,
    controls=[
        ws.num("f1_hz",  "carrier f1 (Hz)",           1000.0, 10000.0, 100.0, 4000.0, fmt=".0f"),
        ws.num("beat_hz", "beat |f1-f2| (Hz, AMF)",      1.0,   250.0,   1.0,  100.0, fmt=".0f"),
        ws.num("ampA",   "channel A amplitude here",     0.0,     1.5,  0.05,    1.0),
        ws.num("ampB",   "channel B amplitude here",     0.0,     1.5,  0.05,    1.0),
    ],
    button="Draw beat",
).show()

# The lab uses f2 - f1 = 2 Hz so contractions are countable by eye; the clinical
# default is nearer 100 Hz. Try both -- panel_beat.run(beat_hz=2) also works.

## Part 3 — The field of four electrodes

### The quadripolar layout

Four electrodes sit on the skin at the corners of a square, wired as two
channels on the two **diagonals**, so their currents cross in the middle:

```
   A+ (top-left)  ........  B+ (top-right)
        \  channel A: A+ -> A-  (one diagonal)
         \                /
          \    x         /   currents cross here
           \            /
   B- (bottom-left) .... A- (bottom-right)
        channel B: B+ -> B-  (other diagonal)
```

### From four electrodes to a modulation map

Each electrode's field is the **same two-layer point-source solution** from
Module 0 (`layered_field`), now evaluated at a horizontal distance
$r=\sqrt{(x-x_e)^2+(y-y_e)^2}$ from an electrode placed anywhere on the surface.
Each channel is a source ($+I_0$) and a sink ($-I_0$), added by superposition:

$$ V_{ch}(x,y,z) = V\big(r_{+},\,z;\,+I_0\big) + V\big(r_{-},\,z;\,-I_0\big). $$

From each channel's potential we take the in-plane current density
$\mathbf{J} = \sigma\mathbf{E}$, $\mathbf{E} = -\nabla V$, giving two vector
fields $\mathbf{a}(x,y)$ (channel A) and $\mathbf{b}(x,y)$ (channel B). Because
the two carriers are only a few Hz apart, both channels see essentially the same
*static* field shape — the interference is entirely in the **time** relationship
between them. Repeating the single-point argument with vectors, the carrier
amplitude at each point swings between $|\mathbf{a}+\mathbf{b}|$ and
$|\mathbf{a}-\mathbf{b}|$, so the **beat modulation depth** is

$$ m(x,y) = \frac{\big|\,|\mathbf{a}+\mathbf{b}| - |\mathbf{a}-\mathbf{b}|\,\big|}{|\mathbf{a}+\mathbf{b}| + |\mathbf{a}-\mathbf{b}|}. $$

- $m = 100\%$ where $\mathbf{a}$ and $\mathbf{b}$ are **collinear and equal**.
- $m = 0\%$ where they are **perpendicular and equal** — which is exactly what
  happens at the geometric centre, where each channel's current runs along its
  own diagonal. So the modulation is *weakest* at the centre and strongest
  off-axis: the four-leaf **clover**.

**Before you run it:** most people expect the "target" of a crossed
four-electrode setup to be the dead centre. Predict where the modulation depth
will actually be highest, then run and check.

**Then try this:**
- Widen or narrow the **pad square** — deeper targeting vs. a tighter, stronger
  pattern.
- Change the **plane depth** — near the surface the pattern is dominated by the
  four pads; deeper down the clover opens up. The **depth section** (second
  figure) shows *all* depths at once, with the purple line marking exactly which
  slice the top-down clover is — watch it move as you change the depth.
- The modulation-depth map is **independent of drive current** (it is a ratio);
  raising the per-channel current only scales the |J| and beat-amplitude maps.
  Confirm that.

In [ ]:
panel_field = ws.Panel(
    ip.draw_ifc_field,
    controls=[
        ws.num("z_depth_mm",          "plane depth (mm)",         2.0,  40.0, 1.0, 15.0),
        ws.num("square_mm",           "pad square side (mm)",    40.0, 120.0, 5.0, 80.0),
        ws.num("sigma1",              "sigma1 skin+fat (S/m)",   0.01,  0.30, 0.01, lf.DEFAULT_SIGMA1),
        ws.num("sigma2",              "sigma2 muscle (S/m)",     0.05,  1.00, 0.01, lf.DEFAULT_SIGMA2),
        ws.num("h_mm",                "shallow layer h (mm)",     1.0,  12.0, 0.5,  lf.DEFAULT_H_MM),
        ws.num("electrode_radius_mm", "pad radius (mm, 0=point)", 0.0,  15.0, 0.5,  0.0),
        ws.num("i0_mA",               "current per channel (mA)", 1.0,  30.0, 1.0, 10.0),
    ],
    button="Draw interferential field",
    note="A few seconds; longer with a non-zero pad radius (each pad becomes a superposition).",
).show()

### Reading the panels

**First figure — top-down (horizontal) view at the chosen depth:**

- **Top row — each channel on its own.** Channel A's current flows between its
  two pads on one diagonal; channel B's between its pads on the other. Each
  looks exactly like Module 0's bipolar field (a source and a sink), just
  oriented along a diagonal — because it *is* that same field.
- **Bottom-left — modulation depth (the clover).** Where the beat is deepest.
  Note the centre is *not* the maximum (the two currents cross at ~90° there);
  the deep-modulation lobes sit off-axis, along the bisectors of the two
  channel diagonals. The dashed/solid contours mark the 50% and 90% depths.
- **Bottom-right — beat current amplitude.** How *strong* the low-frequency beat
  current is (as opposed to how deeply modulated). Comparing this with the
  clover makes an important clinical point: the place with the deepest
  modulation is not necessarily the place with the largest beat current — dose
  and selectivity are two different maps.

**Second figure — perpendicular (depth) section, cutting straight down through
the montage centre.** This is the *penetration* view: the horizontal panels are
a single slice at one depth (the dotted purple line here), while this section
shows the whole depth axis.

- **Left — field penetration.** Current-density magnitude with depth, plus
  current streamlines diving from the surface pads (▼) into the tissue and the
  skin+fat / muscle layer boundary (dashed). Shows how the medium-frequency
  carrier reaches deep structures at all.
- **Right — beat modulation vs depth.** The two strong-modulation lobes reach
  down to the target zone, while a **central null runs straight down the middle**
  (the on-axis point where the two currents stay perpendicular at every depth).
  This is the depth-domain face of the clover, and it uses the *full 3-D* current
  vectors — which is why the on-axis null is exact here.

## Part 4 — Does a fibre actually fire? (NEURON / PyFibers)

Parts 1–3 are pure field/physics. This part puts a **real myelinated axon** in
that field and asks the question that matters clinically: *does interferential
stimulation actually make a nerve fire, and how?* It uses the same MRG
axon model (PyFibers/NEURON) as the transcutaneous workshop's Modules 1–2, so
**it only runs in the WSL environment** (NEURON has no Windows wheel). If you
run the cell below in the Windows environment it will tell you so and skip.

### How the fibre is driven

A straight axon lies in the tissue, inside the four-electrode square. Each
channel's extracellular potential along the fibre is the *same* two-layer field
as Parts 1–3, and the two channels are applied as **two independent PyFibres
sources with their own carriers** — so the potential at each node is

$$ V_e(x,t) = A\,V_A(x)\cos(2\pi f_1 t) + B\,V_B(x)\cos(2\pi f_2 t), $$

the true interferential mix (not a pre-blended waveform). What actually excites
the fibre is the **activating function** $\partial^2 V_e/\partial x^2$ (exactly
as in Module 1) — and because $V_e$ beats at $\Delta f$, so does the drive.

### What to expect

- The membrane is a **low-pass filter**: it cannot follow the 4 kHz carrier
  cycle-by-cycle (you'll see a small subthreshold ripple), but it *integrates*
  the carrier's amplitude. Where the beat swells, the drive crosses threshold
  and the fibre fires; where the beat fades, it goes silent.
- The result is **action-potential bursts locked to the beat frequency** — the
  fibre effectively "hears" the ~100 Hz beat, not the 4 kHz carrier. That is the
  entire point of interferential therapy, shown here from first principles.

**Before you run it:** at low amplitude nothing fires (the kHz carrier is a very
short, charge-balanced pulse — hard to excite with). Predict what happens to the
*number of spikes per beat* as you raise the amplitude.

**Then try this:**
- Raise the amplitude from just-subthreshold upward: watch firing appear as one
  spike per beat, then grow to short bursts per beat (recruitment).
- Change the **beat frequency** (AMF) and confirm the burst *rate* follows it
  while the carrier is unchanged.
- These are model currents in mA; the absolute values are higher than a TENS
  pulse because a kHz sinusoid at depth is an inefficient way to fire a nerve —
  which is itself the reason IFC uses the beat, not the raw carrier.

*(One run is a real nonlinear simulation over many carrier cycles — expect
~15–25 s. Not frozen.)*

In [ ]:
# Carrier, beat and montage size are inherited from Parts 2-3.
panel_ap = ws.Panel(
    ip.draw_ifc_ap,
    controls=[
        ws.choice("diameter", "fibre diameter (um)",
                  [5.7, 7.3, 8.7, 10.0, 11.5, 12.8, 14.0, 15.0, 16.0], 14.0),
        ws.num("amp_mA",          "current per channel (mA)", 0.0, 120.0, 1.0, 65.0),
        ws.num("fiber_depth_mm",  "fibre depth (mm)",         3.0,  20.0, 1.0,  8.0),
    ],
    inherit=["f1_hz", "beat_hz", "square_mm"],
    button="Run NEURON simulation",
    note="~15-25 s: a real nonlinear simulation over many carrier cycles.",
).show()

## Bridge to the lab — the crossed montage on a forearm

Full protocol, safety checklist and recording tables: `handouts/02_interferential_handout.docx`.
The three things the model tells you to watch for:

1. **Both channels must be raised together.** The beat exists only where the two fields overlap
   with comparable amplitude. Modulation depth is 100% where the two channel currents are
   collinear and equal, and **0% where they are perpendicular and equal** — which, as Part 3
   shows, is exactly what happens at the geometric centre of a symmetric crossed montage. The
   deepest modulation sits in four lobes *off* centre, not at the crossing point. This is the
   single most counter-intuitive result of the session, and it contradicts the usual clinical
   picture of the crossing point as the "focus".
2. **Set Δf = 2 Hz for the lab** (CH1 4000 Hz, CH2 4002 Hz) so contractions are countable by eye.
   Then count them over 10 s and check the rate against Δf. Part 2's default of 100 Hz is the
   clinical setting; run both in the notebook so the lab number and the model number are the
   same quantity.
3. **The carrier is 4000 cycles per second and the muscle contracts twice.** That gap is the
   whole point: the membrane is a low-pass filter with a time constant of roughly 0.1-1 ms, so it
   demodulates the envelope and ignores the carrier.

Then detune to Δf = 50 Hz, and finally to Δf = 0, and record what survives.

## Pulling the two notebooks together

| | Notebook 01 — single pulse | Notebook 02 — interferential |
|---|---|---|
| Frequency content | one pulse, 50-250 us | two kHz carriers, beat at Δf |
| What excites the axon | the pulse itself, via d²Ve/dx² | the **envelope** of a carrier too fast to follow |
| Reach into depth | set by pad separation | set by where the two channels overlap |
| Skin comfort | limited by skin impedance at low frequency | carrier crosses skin easily (Z falls with f) |
| Selectivity | pad size and placement | montage geometry — and not where you would guess |
| Cutaneous afferents | recruited first, hence the "bite" | blocked/desensitised under a sustained kHz field |

Both columns are the same volume conductor and the same membrane model. Everything that differs
comes from **when** the current is delivered, not from any new physics — which is the sentence to
leave the session with.

### Discussion to close the electrical-stimulation block
- You measured a chronaxie of a few hundred microseconds in Notebook 01. A 4 kHz carrier has a
  125 us half-cycle. Why does a single 125 us pulse at threshold amplitude fire the axon, while
  4000 of them per second at a comparable amplitude do not fire it 4000 times?
- If the deepest modulation is in off-centre lobes, what does that predict about which fingers
  move — and did the lab agree?
- How would you separate a genuine kHz block of the afferents from habituation or distraction by
  the ongoing contraction? Design one extra control.